In [2]:
import os
import random
from sklearn.model_selection import train_test_split

IMAGE_DIR = "data/processed/originals"
MASK_DIR = "data/processed/binary_masks"

images = sorted(os.listdir(IMAGE_DIR))

train_imgs, temp_imgs = train_test_split(images, test_size=0.30, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=42)

print("Train:", len(train_imgs))
print("Val:", len(val_imgs))
print("Test:", len(test_imgs))

Train: 3500
Val: 750
Test: 750


In [3]:
import torch
from torch.utils.data import Dataset
import cv2
import numpy as np

class SegmentationDataset(Dataset):
    def __init__(self, image_list, image_dir, mask_dir):
        self.image_list = image_list
        self.image_dir = image_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image / 255.0
        image = np.transpose(image, (2, 0, 1))

        mask = cv2.imread(mask_path, 0)
        mask = mask / 255.0
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(image, dtype=torch.float32), \
               torch.tensor(mask, dtype=torch.float32)

In [4]:
from torch.utils.data import DataLoader

train_dataset = SegmentationDataset(train_imgs, IMAGE_DIR, MASK_DIR)
val_dataset = SegmentationDataset(val_imgs, IMAGE_DIR, MASK_DIR)
test_dataset = SegmentationDataset(test_imgs, IMAGE_DIR, MASK_DIR)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [5]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.middle = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        m = self.middle(p3)

        u3 = self.up3(m)
        c3 = self.conv3(torch.cat([u3, d3], dim=1))

        u2 = self.up2(c3)
        c2 = self.conv2(torch.cat([u2, d2], dim=1))

        u1 = self.up1(c2)
        c1 = self.conv1(torch.cat([u1, d1], dim=1))

        return torch.sigmoid(self.final(c1))

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = UNet().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Using device: cpu


In [7]:
def iou_score(pred, target):
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return intersection / (union + 1e-8)

In [8]:
for epoch in range(10):
    model.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)
        loss = criterion(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {train_loss/len(train_loader)}")

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, "checkpoints.pth"
    )
    print("Checkpoint saved for epoch", epoch+1)

KeyboardInterrupt: 

In [11]:
import torch
import os
os.makedirs("checkpoints", exist_ok=True)
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "checkpoints/checkpoints.pth")
print("Checkpoint saved for epoch", epoch+1)

Checkpoint saved for epoch 5


In [9]:
import torch

checkpoint = torch.load("checkpoints/checkpoints.pth", map_location="cpu")

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1

print("Resuming from epoch:", start_epoch)


Resuming from epoch: 5


In [ ]:
num_epochs = 10   # total epochs you originally planned

for epoch in range(start_epoch, num_epochs):

    model.train()
    running_loss = 0

    for images, masks in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch}/{num_epochs}] Loss: {epoch_loss:.4f}")

    # 🔥 Save checkpoint (overwrite same file)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss
    }, "checkpoints/checkpoints.pth")

    print("Checkpoint updated.")


Epoch [5/10] Loss: 0.5093
Checkpoint updated.


In [9]:
import torch

checkpoint = torch.load("checkpoints/checkpoints.pth", map_location="cpu")

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1

print("Resuming from epoch:", start_epoch)

Resuming from epoch: 6


In [ ]:
num_epochs = 10   # total epochs you originally planned

for epoch in range(start_epoch, num_epochs):

    model.train()
    running_loss = 0

    for images, masks in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch}/{num_epochs}] Loss: {epoch_loss:.4f}")

    # 🔥 Save checkpoint (overwrite same file)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss
    }, "checkpoints/checkpoints.pth")

    print("Checkpoint updated.")

Epoch [6/10] Loss: 0.5033
Checkpoint updated.
Epoch [7/10] Loss: 0.4944
Checkpoint updated.


In [8]:
import torch

checkpoint = torch.load("checkpoints/checkpoints.pth", map_location="cpu")

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1

print("Resuming from epoch:", start_epoch)

Resuming from epoch: 8


In [ ]:
num_epochs = 10   # total epochs you originally planned

for epoch in range(start_epoch, num_epochs):

    model.train()
    running_loss = 0

    for images, masks in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch}/{num_epochs}] Loss: {epoch_loss:.4f}")

    # 🔥 Save checkpoint (overwrite same file)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss
    }, "checkpoints/checkpoints.pth")

    print("Checkpoint updated.")

NameError: name 'start_epoch' is not defined

In [22]:
checkpoint = torch.load("checkpoints/checkpoints.pth", map_location="cpu")
print("Saved epoch:", checkpoint['epoch'])

Saved epoch: 0


In [24]:
checkpoint = torch.load("checkpoints/checkpoints.pth", map_location="cpu")

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = 8   

In [25]:
num_epochs = 10   # total epochs you originally planned

for epoch in range(start_epoch, num_epochs):

    model.train()
    running_loss = 0

    for images, masks in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch}/{num_epochs}] Loss: {epoch_loss:.4f}")

    # 🔥 Save checkpoint (overwrite same file)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss
    }, "checkpoints/checkpoints.pth")

    print("Checkpoint updated.")

Epoch [8/10] Loss: 0.5825
Checkpoint updated.
Epoch [9/10] Loss: 0.5467
Checkpoint updated.


In [8]:
import torch

checkpoint = torch.load("checkpoints/checkpoints.pth", map_location="cpu")

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch'] + 1

print("Resuming from epoch:", start_epoch)


Resuming from epoch: 10


In [9]:
num_epochs = 10   # total epochs you originally planned

for epoch in range(start_epoch, num_epochs):

    model.train()
    running_loss = 0

    for images, masks in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch}/{num_epochs}] Loss: {epoch_loss:.4f}")

    # 🔥 Save checkpoint (overwrite same file)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss
    }, "checkpoints/checkpoints.pth")

    print("Checkpoint updated.")